# ORena FOCUS — FRAME Track Pipeline
**Single-image surgical VQA | HeiCo dataset | Qwen2-VL**

Run top-to-bottom on first pass. Each section can be re-run independently after setup.

```
[ ] 0. Environment Setup
[ ] 1. Dataset Loading
[ ] 2. Ontology & Data Analysis
[ ] 3. Frame Extraction (video → image)
[ ] 4. Model Loading
[ ] 5. Inference & Prompt Engineering
[ ] 6. Evaluation & Error Analysis
[ ] 7. Batch Evaluation
[ ] 8. Iteration & Experiments
[ ] 9. Results
```

---
## 0. Environment Setup
*Re-run at the start of every Kaggle session.*

In [ ]:
import os, sys

os.system('pip install -q orena-focus transformers accelerate qwen-vl-utils huggingface_hub')

# Clone or update repo
if not os.path.exists('/kaggle/working/orena-focus-vqa'):
    os.system('git clone https://github.com/Blue0318/orena-focus-vqa.git /kaggle/working/orena-focus-vqa')
    print('Repo cloned')
else:
    os.system('cd /kaggle/working/orena-focus-vqa && git pull -q')
    print('Repo updated')

sys.path.append('/kaggle/working/orena-focus-vqa/src')

# HF token
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded')
except Exception as e:
    print(f'Warning: No HF_TOKEN — {e}')

print('\nEnvironment ready')

---
## 1. Dataset Loading

In [ ]:
from focus import FocusDataset, DatasetSplit, Track

ds = FocusDataset('heico', DatasetSplit.TRAIN, Track.FRAME)
print(f'Train samples: {len(ds)}')

req, ref = ds[0]
print(f'\nSample question : {req.question}')
print(f'Ground truth    : {ref.answer}')
print(f'Answer format   : {ref.format.type}')
print(f'Video ID        : {req.videoID}')
print(f'Start time (ms) : {req.start_time}')

---
## 2. Ontology & Data Analysis

In [ ]:
from collections import Counter, defaultdict
import matplotlib.pyplot as plt

all_refs = [ref for _, ref in ds]

# Format distribution
formats = [str(ref.format.type) for ref in all_refs]
counts  = Counter(formats)

print('Format distribution:')
for fmt, n in counts.most_common():
    print(f'  {fmt:<20} {n:>5}  ({100*n/len(all_refs):.1f}%)')

plt.figure(figsize=(10, 4))
plt.bar(counts.keys(), counts.values(), color='steelblue')
plt.title('FRAME Track — Answer Format Distribution')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# FO class vocabulary
fo_answers = [ref.answer for ref in all_refs if str(ref.format.type) == 'fo_class']
fo_counts  = Counter(fo_answers)

print(f'Unique FO classes: {len(fo_counts)}')
for cls, n in fo_counts.most_common():
    bar = '█' * (n // 20)
    print(f'  {cls:<40} {n:>4}  {bar}')

# Multi-label ordering
multi      = [ref.answer for ref in all_refs if ',' in ref.answer]
not_sorted = [a for a in multi if a.split(', ') != sorted(a.split(', '))]
print(f'\nMulti-label answers    : {len(multi)}')
print(f'Not alphabetically sorted: {len(not_sorted)}')
for a in not_sorted[:5]:
    print(f'  "{a}"')

In [ ]:
# Sample Q&A per format — know what your model must handle
samples = defaultdict(list)
for req_i, ref_i in ds:
    fmt = str(ref_i.format.type)
    if len(samples[fmt]) < 2:
        samples[fmt].append((req_i.question, ref_i.answer))

for fmt, pairs in sorted(samples.items()):
    print(f'\n[{fmt.upper()}]')
    for q, a in pairs:
        print(f'  Q: {q}')
        print(f'  A: {repr(a)}')

---
## 3. Frame Extraction
*Videos live at `orena-dkfz/heico-focus-vqa/videos/<videoID>` on HuggingFace.*
*`hf_hub_download` fetches and caches them locally — same video not re-downloaded.*

In [ ]:
import cv2
import numpy as np
from PIL import Image
from huggingface_hub import hf_hub_download

# Cache: video_id → local path (avoid re-downloading within session)
_video_cache = {}

def get_video_path(video_id: str) -> str:
    """Download video from HF (cached after first download)."""
    if video_id not in _video_cache:
        path = hf_hub_download(
            repo_id='orena-dkfz/heico-focus-vqa',
            filename=f'videos/{video_id}',
            repo_type='dataset',
        )
        _video_cache[video_id] = path
    return _video_cache[video_id]


def extract_frame(video_id: str, start_time_ms: float) -> Image.Image:
    """Extract single frame at start_time_ms and return as PIL Image."""
    path  = get_video_path(video_id)
    cap   = cv2.VideoCapture(path)
    fps   = cap.get(cv2.CAP_PROP_FPS)
    frame_idx = int((start_time_ms / 1000) * fps)

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()

    if not ret:
        raise ValueError(f'Could not read frame {frame_idx} from {video_id}')

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return Image.fromarray(frame_rgb)


# Test on sample 0
req, ref = ds[0]
frame = extract_frame(req.videoID, req.start_time)

plt.figure(figsize=(8, 6))
plt.imshow(frame)
plt.title(f'Q: {req.question}\nA: {ref.answer}')
plt.axis('off')
plt.show()
print(f'Frame size: {frame.size}')

---
## 4. Model Loading

In [ ]:
import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU  :', torch.cuda.get_device_name(0))
    print('VRAM :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Start with 2B for speed — swap to 7B when pipeline is stable
MODEL_NAME = 'Qwen/Qwen2-VL-2B-Instruct'

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model     = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model.eval()
print(f'Model loaded: {MODEL_NAME}')

---
## 5. Inference & Prompt Engineering
*Prompt logic lives in `src/focus_vqa/prompts/templates.py`.*
*Edit that file and re-import — do not duplicate prompt code here.*

In [ ]:
from focus_vqa.prompts.templates import build_prompt, normalize_fo_class

# Constrained vocab — hardcoded from dataset inspection
VALID_CLASSES = [
    'Clip', 'External drain', 'Needle',
    'Silicone loop', 'Specimen', 'Specimen bag', 'Sponge',
]
MAX_NEW_TOKENS = 30


def run_inference(model, processor, frame: Image.Image, prompt: str) -> str:
    """Run Qwen2-VL inference on a single frame + prompt."""
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': frame},
            {'type': 'text',  'text':  prompt},
        ],
    }]

    text   = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[frame], return_tensors='pt').to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )

    trimmed = output_ids[:, inputs['input_ids'].shape[1]:]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()


# Test on sample 0
req, ref = ds[0]
frame    = extract_frame(req.videoID, req.start_time)
prompt   = build_prompt(req)

print('PROMPT:')
print(prompt)
print()

prediction = run_inference(model, processor, frame, prompt)

print('PREDICTION :', prediction)
print('GROUND TRUTH:', ref.answer)
print('MATCH       :', prediction.strip().lower() == ref.answer.strip().lower())

---
## 6. Evaluation & Error Analysis

In [ ]:
# Simple evaluator — extend with orena Evaluator once batch results are stable
def evaluate(prediction: str, ground_truth: str, fmt: str) -> dict:
    pred_norm = normalize_fo_class(prediction.strip()) if fmt == 'fo_class' else prediction.strip()
    gt_norm   = ground_truth.strip()
    return {
        'prediction'  : pred_norm,
        'ground_truth': gt_norm,
        'format'      : fmt,
        'correct'     : pred_norm.lower() == gt_norm.lower(),
    }


# Single sample evaluation
req, ref = ds[0]
frame    = extract_frame(req.videoID, req.start_time)
prompt   = build_prompt(req)
pred     = run_inference(model, processor, frame, prompt)
result   = evaluate(pred, ref.answer, str(ref.format.type))

print(f'Format      : {result["format"]}')
print(f'Question    : {req.question}')
print(f'Prediction  : {result["prediction"]}')
print(f'Ground truth: {result["ground_truth"]}')
print(f'Correct     : {result["correct"]}')

---
## 7. Batch Evaluation

In [ ]:
import pandas as pd

# Adjust N — start small (20), scale up once stable
N = 20
results = []

for idx in range(N):
    try:
        req_i, ref_i = ds[idx]
        frame_i      = extract_frame(req_i.videoID, req_i.start_time)  # cached after first download
        prompt_i     = build_prompt(req_i)
        pred_i       = run_inference(model, processor, frame_i, prompt_i)
        result_i     = evaluate(pred_i, ref_i.answer, str(ref_i.format.type))
        result_i['idx'] = idx
        result_i['question'] = req_i.question
        results.append(result_i)

        symbol = '✓' if result_i['correct'] else '✗'
        print(f'[{idx:02d}] {symbol}  pred={repr(pred_i):<30}  gt={repr(ref_i.answer)}')

    except Exception as e:
        print(f'[{idx:02d}] ERROR: {e}')

df = pd.DataFrame(results)
acc = df['correct'].mean()
print(f'\nOverall accuracy : {acc:.1%}  ({df["correct"].sum()}/{len(df)})')
print('\nAccuracy by format:')
print(df.groupby('format')['correct'].agg(['mean', 'count']).round(3))

In [ ]:
# Error analysis — look at failures
failures = df[df['correct'] == False]
print(f'Failures: {len(failures)}')
for _, row in failures.iterrows():
    print(f'\n[{row["idx"]:02d}] [{row["format"]}]')
    print(f'  Q   : {row["question"]}')
    print(f'  Pred: {repr(row["prediction"])}')
    print(f'  GT  : {repr(row["ground_truth"])}')

---
## 8. Iteration & Experiments
*Log every change here. If score doesn't improve, revert.*
*Edit `src/focus_vqa/prompts/templates.py`, re-import, re-run batch eval.*

In [ ]:
# Experiment log
experiments = [
    # {'exp': 'baseline', 'model': 'Qwen2-VL-2B', 'notes': 'default prompt', 'accuracy': None},
]

if experiments:
    print(pd.DataFrame(experiments).to_string())
else:
    print('No experiments logged yet')

In [ ]:
# Reload prompt module after editing templates.py
import importlib
import focus_vqa.prompts.templates as tpl
importlib.reload(tpl)
from focus_vqa.prompts.templates import build_prompt, normalize_fo_class
print('Templates reloaded')

---
## 9. Results

In [ ]:
if 'df' in dir() and len(df) > 0:
    print('=== FINAL RESULTS ===')
    print(f'Samples evaluated : {len(df)}')
    print(f'Overall accuracy  : {df["correct"].mean():.1%}')
    print()
    print('By format:')
    print(df.groupby('format')['correct'].agg(['mean','count']).round(3))

    # Save to results/
    out = '/kaggle/working/orena-focus-vqa/results/frame'
    os.makedirs(out, exist_ok=True)
    df.to_csv(f'{out}/results.csv', index=False)
    print(f'\nSaved to {out}/results.csv')
else:
    print('Run batch evaluation first (Section 7)')